In [7]:
import pandas as pd
import numpy as np
from sklearn.utils import shuffle
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
# Load data
X_train_resampled = pd.read_csv("X_train_resampled.csv")
y_train_resampled = pd.read_csv("y_train_resampled.csv").values.ravel()

X_test = pd.read_csv("X_test.csv")
y_test = pd.read_csv("y_test.csv").values.ravel()

print("Training Set Size:", X_train_resampled.shape)
print("Test Set Size:", X_test.shape)

Training Set Size: (454902, 29)
Test Set Size: (56962, 29)


In [ ]:
Data Sampling for Experimentation

In [ ]:
X_sample, y_sample = shuffle(X_train_resampled, y_train_resampled, random_state=42)
X_sample = X_sample[:10000]
y_sample = y_sample[:10000]

print("Sampled Training Size:", X_sample.shape)

Sampled Training Size: (10000, 29)


In [ ]:
Tried multiple parameters
Linear kernel with different C
RBF kernel with different gamma

In [ ]:
#Experimenting with Linear Kernel
C_values = [0.1, 1, 10]
for C in C_values:
    model = SVC(kernel='linear', C=C)
    model.fit(X_sample, y_sample)
    y_pred = model.predict(X_test)
    print(f"\nLinear Kernel — C={C}")
    print(f"Recall (Fraud): {classification_report(y_test, y_pred, output_dict=True)['1']['recall']:.4f}")
    print(f"Precision (Fraud): {classification_report(y_test, y_pred, output_dict=True)['1']['precision']:.4f}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")


Linear Kernel — C=0.1
Recall (Fraud): 0.9184
Precision (Fraud): 0.0573
Accuracy: 0.9739

Linear Kernel — C=1
Recall (Fraud): 0.9184
Precision (Fraud): 0.0535
Accuracy: 0.9719

Linear Kernel — C=10
Recall (Fraud): 0.9184
Precision (Fraud): 0.0521
Accuracy: 0.9711


In [19]:
#Experimenting with RBF (Gaussian) Kernel
gamma_values = [0.01, 0.1, 1]
C_fixed = 1.0
for gamma in gamma_values:
    model = SVC(kernel='rbf', C=C_fixed, gamma=gamma)
    model.fit(X_sample, y_sample)
    y_pred = model.predict(X_test)
    print(f"\nRBF Kernel — C={C_fixed}, gamma={gamma}")
    print(f"Recall (Fraud): {classification_report(y_test, y_pred, output_dict=True)['1']['recall']:.4f}")
    print(f"Precision (Fraud): {classification_report(y_test, y_pred, output_dict=True)['1']['precision']:.4f}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")



RBF Kernel — C=1.0, gamma=0.01
Recall (Fraud): 0.9184
Precision (Fraud): 0.0792
Accuracy: 0.9815

RBF Kernel — C=1.0, gamma=0.1
Recall (Fraud): 0.7347
Precision (Fraud): 0.1622
Accuracy: 0.9930

RBF Kernel — C=1.0, gamma=1
Recall (Fraud): 0.1837
Precision (Fraud): 0.2400
Accuracy: 0.9976


In [ ]:
Final Evaluation

In [20]:
# Training the final model with best params (kernel: RBF, C=1, gamma=0.1)
X_final_train, y_final_train = shuffle(X_train_resampled, y_train_resampled, random_state=42)
X_final_train = X_final_train[:20000]
y_final_train = y_final_train[:20000]

final_model = SVC(kernel='rbf', C=1.0, gamma=0.1, probability=True)
final_model.fit(X_final_train, y_final_train)

y_pred_final = final_model.predict(X_test)
y_prob_final = final_model.predict_proba(X_test)[:, 1]

# Final Evaluation
print("\n--- Final Model Evaluation (RBF Kernel) ---")
print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_final))
print("Classification Report:\n", classification_report(y_test, y_pred_final))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_final))


--- Final Model Evaluation (RBF Kernel) ---
Accuracy: 0.9945753309223693
Confusion Matrix:
 [[56587   277]
 [   32    66]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.19      0.67      0.30        98

    accuracy                           0.99     56962
   macro avg       0.60      0.83      0.65     56962
weighted avg       1.00      0.99      1.00     56962

ROC-AUC: 0.9731023107048109
